# Lab 06 — AI_SUMMARIZE & AI_TRANSLATE

**Summarize** condenses long text. **Translate** converts between languages. Natural partners for global content.

| Function | Returns | Use Case |
|---|---|---|
| `AI_SUMMARIZE` | Concise summary text | Executive digests, knowledge bases |
| `AI_TRANSLATE` | Translated text | Multilingual catalogs, global support |


## Step 1 — Environment Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — AI_SUMMARIZE: Article Summarization

In [ ]:
CREATE OR REPLACE TABLE wiki_articles (
    article_id INT AUTOINCREMENT, title VARCHAR(200),
    category VARCHAR(100), word_count INT, body TEXT
);

INSERT INTO wiki_articles (title, category, word_count, body) VALUES
('Snowflake (company)', 'Technology', 420, 'Snowflake Inc. is a cloud computing company that provides a cloud-based data warehousing platform. Founded in 2012 by Benoit Dageville, Thierry Cruanes, and Marcin Zukowski, the company is headquartered in Bozeman, Montana. Snowflake''s platform allows businesses to store, manage, and analyze large volumes of structured and semi-structured data using a cloud-native architecture. The platform separates compute from storage, enabling users to scale each independently. Snowflake supports multiple cloud providers including Amazon Web Services, Microsoft Azure, and Google Cloud Platform. The company went public in September 2020 in what was then the largest software IPO in history. Snowflake Cortex brings AI and ML capabilities directly into the data platform, allowing users to build intelligent applications without moving data.'),
('Large language model', 'Artificial Intelligence', 380, 'A large language model (LLM) is a type of artificial intelligence model trained on vast amounts of text data to understand and generate human-like text. These models use transformer architectures with billions of parameters. Training involves processing terabytes of text from the internet, books, and other sources. Key concepts include tokenization (breaking text into units), attention mechanisms (focusing on relevant context), and fine-tuning (adapting to specific tasks). Popular LLMs include GPT-4, Claude, Llama, and Mistral. LLMs can perform tasks like text generation, summarization, translation, classification, and question answering. Challenges include hallucination (generating plausible but false information), bias in training data, and high computational costs.'),
('Retrieval-Augmented Generation', 'Artificial Intelligence', 350, 'Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. Instead of relying solely on a language model''s training data, RAG first retrieves relevant documents from a knowledge base, then uses the retrieved context to generate more accurate and grounded responses. The process involves converting documents into vector embeddings, storing them in a vector database, and at query time, finding the most relevant documents using similarity search. RAG reduces hallucination by grounding responses in actual data.');

In [ ]:
SELECT
    title, category, word_count,
    SNOWFLAKE.CORTEX.AI_SUMMARIZE(body) AS summary
FROM wiki_articles;

In [ ]:
WITH combined AS (
    SELECT category,
        LISTAGG(body, '\n\n') WITHIN GROUP (ORDER BY title) AS all_text
    FROM wiki_articles
    GROUP BY category
)
SELECT category, SNOWFLAKE.CORTEX.AI_SUMMARIZE(all_text) AS category_digest
FROM combined;

## Step 3 — AI_TRANSLATE: Multilingual Content

In [ ]:
CREATE OR REPLACE TABLE product_catalog_multilingual (
    entry_id INT AUTOINCREMENT, sku VARCHAR(20),
    category VARCHAR(100), language VARCHAR(10), description TEXT
);

INSERT INTO product_catalog_multilingual (sku, category, language, description) VALUES
('SKU-001', 'Electronics', 'en', 'High-performance wireless headphones with active noise cancellation and 30-hour battery life.'),
('SKU-001', 'Electronics', 'es', 'Auriculares inalambricos de alto rendimiento con cancelacion activa de ruido y 30 horas de bateria.'),
('SKU-001', 'Electronics', 'fr', 'Casque sans fil haute performance avec reduction active du bruit et 30 heures de batterie.'),
('SKU-001', 'Electronics', 'de', 'Hochleistungs-Funkkopfhorer mit aktiver Gerauschunterdruckung und 30 Stunden Akkulaufzeit.'),
('SKU-002', 'Kitchen', 'en', 'Smart coffee maker with built-in grinder and WiFi connectivity. Makes up to 12 cups.'),
('SKU-002', 'Kitchen', 'ja', 'ビルトイングラインダーとWiFi接続を備えたスマートコーヒーメーカー。最大12杯まで抽出可能。'),
('SKU-002', 'Kitchen', 'zh', '内置研磨机和WiFi连接的智能咖啡机。最多可冲泡12杯。');

In [ ]:
SELECT
    sku, category, language AS source_lang,
    SNOWFLAKE.CORTEX.AI_TRANSLATE(description, language, 'en') AS english_description
FROM product_catalog_multilingual
WHERE language != 'en'
ORDER BY sku;

## Step 4 — Combining: Translate then Summarize

In [ ]:
SELECT
    sku,
    SNOWFLAKE.CORTEX.AI_SUMMARIZE(
        SNOWFLAKE.CORTEX.AI_TRANSLATE(description, language, 'en')
    ) AS english_summary
FROM product_catalog_multilingual
WHERE language != 'en'
LIMIT 3;

## Key Takeaways

- `AI_SUMMARIZE` handles long text — combine with `LISTAGG` for multi-row summaries.
- `AI_TRANSLATE` supports 10+ languages in a single function call.
- Chain functions: translate → then summarize, or summarize → then translate.
- Both work on any text column — articles, emails, catalog entries, support tickets.
